# 03 — Spark Query Playground

Uses `SparkFactory` to read price snapshots and OHLCV bars directly from MinIO via S3A.
Good for cross-day, cross-symbol queries that would be slow in plain Python.

**Requires:** MinIO running with data; `SPARK_MASTER_URL` unset (uses `local[*]`) or set to the cluster URL in `.env`.

> **Note:** First run downloads no JARs — they are pre-baked in the Docker image when running in cluster mode, and resolved from the venv's PySpark distribution in local mode. Local mode does require `hadoop-aws` and `aws-java-sdk` JARs on `SPARK_HOME/jars`; if you see S3A errors, run the Spark Docker cluster and set `SPARK_MASTER_URL=spark://spark-master:7077` in `.env` instead.

In [ ]:
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from model.spark import SparkFactory
from model.minio_store import MinioStore

load_dotenv()
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
# SparkFactory is a context manager — use `with` so the session is stopped cleanly.
# In a notebook we keep it open for the whole session instead.
factory = SparkFactory('notebook')
spark   = factory.session   # lazy — session starts here
print('Spark master:', spark.sparkContext.master)
print('Spark version:', spark.version)

In [ ]:
# List today's Avro files from MinIO and load them into a Spark DataFrame
from datetime import date

raw_store = MinioStore(os.getenv('MINIO_BUCKET', 'market-data'))

today = date.today()
date_frag = f'/year={today.year}/month={today.strftime("%m")}/day={today.strftime("%d")}/'

snapshot_files = [
    f's3a://{raw_store.bucket}/{obj.object_name}'
    for obj in raw_store.list_objects(prefix='price.snapshot/')
    if date_frag in obj.object_name and obj.object_name.endswith('.avro')
]
print(f'Found {len(snapshot_files)} Avro files for {today}')

In [ ]:
if not snapshot_files:
    print('No files found. Make sure the storage consumer has been running today.')
else:
    df_snaps = spark.read.format('avro').load(snapshot_files)
    df_snaps.printSchema()
    print(f'{df_snaps.count():,} rows')

In [ ]:
# Tick count per symbol — how active was each symbol today?
tick_counts = (
    df_snaps
    .groupBy('symbol')
    .agg(
        F.count('*').alias('ticks'),
        F.round(F.min('price'), 2).alias('low'),
        F.round(F.max('price'), 2).alias('high'),
        F.round(F.avg('price'), 2).alias('avg_price'),
        F.max('volume').alias('total_volume'),
    )
    .orderBy('ticks', ascending=False)
)
tick_counts.show()

In [ ]:
# Intra-day price range per symbol (as a percentage of avg price)
range_df = (
    tick_counts.withColumn(
        'range_pct',
        F.round((F.col('high') - F.col('low')) / F.col('avg_price') * 100, 2)
    )
    .select('symbol', 'low', 'high', 'avg_price', 'range_pct')
    .orderBy('range_pct', ascending=False)
)
range_df.show()

In [ ]:
# Load ALL OHLCV bars across all dates from market-analysis
analysis_store = MinioStore(os.getenv('MINIO_ANALYSIS_BUCKET', 'market-analysis'))

ohlcv_files = [
    f's3a://{analysis_store.bucket}/{obj.object_name}'
    for obj in analysis_store.list_objects(prefix='ohlcv.bar/')
    if obj.object_name.endswith('.parquet')
]
print(f'Found {len(ohlcv_files)} OHLCV Parquet files')

In [ ]:
if ohlcv_files:
    df_ohlcv = spark.read.parquet(*ohlcv_files)
    df_ohlcv.printSchema()

    # 7-day rolling average close per symbol using Spark window functions
    from pyspark.sql.window import Window

    w = Window.partitionBy('symbol').orderBy('time').rowsBetween(-6, 0)
    df_with_sma = (
        df_ohlcv
        .withColumn('sma7', F.round(F.avg('close').over(w), 2))
        .select('time', 'symbol', 'close', 'sma7')
        .orderBy('symbol', 'time')
    )
    df_with_sma.show(20)

In [ ]:
# Visualise close vs SMA7 for one symbol using Pandas after Spark aggregation
if ohlcv_files:
    import pandas as pd
    SYMBOL = df_ohlcv.select('symbol').distinct().collect()[0][0]

    pdf = (
        df_with_sma
        .filter(F.col('symbol') == SYMBOL)
        .toPandas()
    )
    pdf['date'] = pd.to_datetime(pdf['time'].str[:10])

    fig, ax = plt.subplots()
    ax.plot(pdf['date'], pdf['close'], label='Close', linewidth=1.5)
    ax.plot(pdf['date'], pdf['sma7'],  label='SMA 7', linestyle='--', linewidth=1)
    ax.set_title(f'{SYMBOL} — Close vs SMA 7 (Spark)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Stop the Spark session when done to free resources
factory.stop()